# 06 — Población en riesgo: salida y caída en mayor informalidad (diagnóstico)

El eje del TFI es predecir, a nivel hogar, el riesgo de **caer** en mayor
informalidad laboral o de **no salir** de ella, según el quintil de ingreso y las
condiciones habitacionales. 

Este notebook mide las **dos direcciones** del resultado, con el mismo tipo de
diagnóstico, **sin ajustar ningún modelo todavía**:
- **Salida** de la precariedad (implementada): define 3 candidatas de "hogar en riesgo"
  en `t`, mide su salida en `t+h` (`h=1,4`, sobre `data/03_panel/pares_h1`/`pares_h4`,
  `identidad_confirmada==True`), agrega una variante estricta con las trayectorias de 4
  apariciones del notebook 04 (`data/04_trayectorias/`). **No elige una definición** --
  deja las 3 con sus números reales para la entrada D14.
- **Caída** en mayor informalidad (diseño elegido con Iván, todavía sin implementar
  en este notebook): intensificación de `share_precario = (asalariados_informales +
  desocupados) / activos` dentro del hogar, sobre personas activas emparejadas entre `t`
  y `t+h` -- diseño completo, por qué se descartó la alternativa más simple y lo
  pendiente en D16 (`docs/decisiones_metodologicas.md`).

Arma también el inventario de predictores disponibles en `t`.

**Revisión de Iván sobre la primera corrida** -- 4 correcciones incorporadas:
1. El bootstrap de la sección 2b tenía un bug real (IC artificialmente angosto) -- el
   mecanismo se extrajo a `src/bootstrap.py` (con tests) para poder testearlo.
2. "Salida" se redefine en positivo, **distinto por definición** (sección 2): qué se
   abandona en `a`/`c` (ingreso) no es lo mismo que en `b` (solo empleo). Pasar a no
   tener ningún miembro activo deja de contarse como salida -- queda en categoría aparte,
   con una variante principal (cuenta como "no sale") y una de robustez (se excluye).
3. La comparación por `ancla_con_deficit` se repite estratificando por decil y con una
   versión estandarizada (sección 3) -- cuánto de la brecha sobrevive dentro de cada
   decil vs. cuánto es composición.
4. La familia de deciles en `t` se saca del bloque "excluido por fuga" del inventario
   (sección 4) -- la fuga es cualquier variable de `t+h`, no la posición de ingreso en
   `t`.

## Cautelas

- **Quintiles/deciles como posición relativa**: `DECCFR` no depende de deflactar con el
  IPC intervenido de 2007-2015 -- dos de las tres definiciones candidatas lo usan sin ese
  problema.
- **D3 (ingreso no declarado)**: desde 2016, `DECCFR==12`/`PONDIH==0` deja sin quintil
  válido a una fracción grande de hogares (~28.5% en la era regular, notebook 05). Afecta
  dos veces acá: a la población en riesgo de a/c en `t`, y al `decil_th` necesario para
  decidir la categoría de salida en `t+h` -- la sección 2b-bis mide si esa segunda falta
  de dato es pareja entre hogares con y sin déficit habitacional.
- **D13 (ruptura de `CODUSU` 2013T1/2013T2)**: los pares `h=4` con origen en
  `{2013T1,2013T2,2013T3}` están rotos o truncados a la mitad de la población -- se
  excluyen de toda tasa de salida `h=4` y de la variante con trayectorias, reusando el
  flag `corte_estructural_d13` ya calculado en
  `data/meta/trayectorias_completitud_cohorte.csv`. `h=1` no se ve afectado.
- **Rotación 2-2-2 del panel**: la "visita siguiente" real de un hogar después de `t0+1`
  es `t0+4`, no `t0+2` (no existe) -- la variante estricta de salida usa esa visita real.
- Sin requerimiento de gráficos (a diferencia del 05) -- salida tabular/CSV.

Lee `data/02_nucleo/{hogar,individual}`, `data/01_armonizado/{hogar,individual}`,
`data/03_panel/pares_h1`/`pares_h4` y `data/04_trayectorias/trayectorias/` +
`data/meta/trayectorias_completitud_cohorte.csv`. Importa `src/bootstrap.py` (único
módulo de este repo que un notebook importa en vez de duplicar -- duplicar un mecanismo
de inferencia estadística con un bug ya encontrado una vez es más riesgoso que duplicar
una regla de negocio). No escribe ninguna capa `data/0N_*` -- solo CSV en `data/meta/`.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
from perfil_hogar import calcular_grupo_ingreso, calcular_situacion_laboral, calcular_decil, to_bool, kleene_any_hogar, calcular_componentes_ancla
from constantes import *
from armonizacion import trimestres_publicados, es_historico
import panel
import pipeline_meta as pm
from bootstrap import diferencia_dos_grupos_bootstrap_cluster, estandarizar_diferencia_bootstrap

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional de orígenes t; None = todos

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
PANEL = REPO / "data" / "03_panel"
TRAYECTORIAS = REPO / "data" / "04_trayectorias"
META = REPO / "data" / "meta"

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_a_procesar = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
SUFIJO = "_parcial" if ES_CORRIDA_PARCIAL else ""
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))

COHORTE_D13 = pd.read_csv(META / "trayectorias_completitud_cohorte.csv")
ORIGENES_EXCLUIDOS_D13 = set(
    COHORTE_D13.loc[COHORTE_D13["corte_estructural_d13"] == True, ["ANO4_t0", "TRIMESTRE_t0"]]
    .itertuples(index=False, name=None)
)
print(f"orígenes excluidos por D13 (h=4 y trayectorias): {sorted(ORIGENES_EXCLUIDOS_D13)}")


`cargar_hogar`/`cargar_individual` se importan de `pipeline_meta` (uso general,
compartido con 03/04/05) -- no se duplican acá; esta notebook no pasa `ARMONIZADO`
a `cargar_individual`, así que no trae `NIVEL_ED` (no lo usa).

## Funciones reusadas (duplicadas de 03/05) y `calcular_precariedad`

`calcular_grupo_ingreso`, `calcular_situacion_laboral` y `calcular_componentes_ancla`
están duplicadas tal cual de `03_exploracion_descriptiva.ipynb`. `to_bool`/
`kleene_any_hogar` están duplicadas de `05_composicion_quintiles.ipynb` (NA real vía
lógica de Kleene, no `False` silencioso).

`calcular_precariedad` combina todo, incluidos los insumos que ahora necesita la
categorización de salida de la sección 2: `decil` (entero 1-10), `algun_activo` y
`algun_formal` (antes intermedios sin exponer).

In [ ]:
COLUMNAS_HOGAR = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "DECCFR", "PONDIH", "PONDERA",
                  "II1", "II2", "II5", "II5_1", "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3"]
COLUMNAS_INDIVIDUAL = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE",
                        "CH03", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PONDERA"]

DEFINICIONES = ["a", "b", "c"]
DESCRIPCION_DEFINICION = {
    "a": "DECCFR en Q1-Q2 y activos_sin_empleo_formal",
    "b": "activos_sin_empleo_formal (sin condición de ingreso)",
    "c": "DECCFR en Q1 solo",
}


def calcular_precariedad(hogar: pd.DataFrame, individual: pd.DataFrame) -> pd.DataFrame:
    '''Por hogar: grupo_ingreso, decil, ancla_con_deficit, activos_sin_empleo_formal,
    algun_activo, algun_formal, situacion_laboral y los 3 flags precario_a/b/c
    (nullable boolean, NA real si falta el insumo).'''
    idx = hogar.set_index(["CODUSU", "NRO_HOGAR"]).index
    h = hogar.set_index(["CODUSU", "NRO_HOGAR"])
    grupo = calcular_grupo_ingreso(hogar); grupo.index = idx
    decil = calcular_decil(hogar); decil.index = idx

    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().reindex(idx)
    ancla = calcular_componentes_ancla(h, n_miembros)["ancla_con_deficit"]

    situacion = calcular_situacion_laboral(individual).set_index(["CODUSU", "NRO_HOGAR"])["situacion_laboral"].reindex(idx)
    algun_formal = (situacion == "algun_asalariado_formal").astype("boolean")
    algun_formal[situacion.isna()] = pd.NA
    ningun_formal = (situacion != "algun_asalariado_formal").astype("boolean")
    ningun_formal[situacion.isna()] = pd.NA
    algun_activo = kleene_any_hogar(individual, to_bool(individual["ESTADO"], [1, 2])).reindex(idx)
    activos_sin_empleo_formal = (algun_activo & ningun_formal).astype("boolean")
    activos_sin_empleo_formal[algun_activo.isna() | (algun_activo.fillna(False) & ningun_formal.isna())] = pd.NA

    q1q2 = grupo.isin(["Q1", "Q2"]).astype("boolean")
    q1q2[grupo.isna()] = pd.NA
    q1 = (grupo == "Q1").astype("boolean")
    q1[grupo.isna()] = pd.NA

    precario_a = (q1q2 & activos_sin_empleo_formal).astype("boolean")
    precario_a[q1q2.isna() | activos_sin_empleo_formal.isna()] = pd.NA
    precario_b = activos_sin_empleo_formal
    precario_c = q1

    return pd.DataFrame({
        "grupo_ingreso": grupo, "decil": decil, "ancla_con_deficit": ancla,
        "activos_sin_empleo_formal": activos_sin_empleo_formal, "algun_activo": algun_activo,
        "algun_formal": algun_formal, "situacion_laboral": situacion,
        "precario_a": precario_a, "precario_b": precario_b, "precario_c": precario_c,
    }, index=idx)

## Caché de precariedad por trimestre

`calcular_precariedad` se corre **una sola vez por trimestre** y se guarda en
`CACHE_PRECARIEDAD`, indexado por `(anio, trimestre)` -- todas las secciones lo reusan
sin volver a leer `02_nucleo/`.

In [ ]:
def trimestres_destino_necesarios(origenes: list[tuple[int, int]]) -> set[tuple[int, int]]:
    destinos = set(origenes)
    for anio, trimestre in origenes:
        for offset in (1, 4, 5):
            destinos.add(panel.sumar_trimestres(anio, trimestre, offset))
    return destinos & set(TODOS_LOS_TRIMESTRES)


TRIMESTRES_NECESARIOS = sorted(trimestres_destino_necesarios(trimestres_a_procesar))

t0 = time.time()
CACHE_PRECARIEDAD: dict[tuple[int, int], pd.DataFrame] = {}
n_filas_total = 0
for anio, trimestre in TRIMESTRES_NECESARIOS:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO)
    if hogar is None or individual is None:
        continue
    prec = calcular_precariedad(hogar, individual)
    CACHE_PRECARIEDAD[(anio, trimestre)] = prec
    n_filas_total += len(prec)
print(f"{len(CACHE_PRECARIEDAD)} trimestres en caché, {n_filas_total:,} hogares ({time.time()-t0:.1f}s)")

## Sección 1 — Población en riesgo en `t`

`N` = cantidad de hogares (sin ponderar) que cumplen cada definición, por trimestre y
era, sobre **todos** los hogares de `02_nucleo/hogar` en ese trimestre.

In [ ]:
filas_poblacion = []
for (anio, trimestre) in trimestres_a_procesar:
    prec = CACHE_PRECARIEDAD.get((anio, trimestre))
    if prec is None:
        continue
    era = "historica" if es_historico(anio, trimestre) else "regular"
    fila = {"anio": anio, "trimestre": trimestre, "era": era, "n_hogares_total": len(prec)}
    for d in DEFINICIONES:
        col = prec[f"precario_{d}"]
        fila[f"n_precario_{d}"] = int((col == True).sum())
        fila[f"n_na_{d}"] = int(col.isna().sum())
    filas_poblacion.append(fila)

riesgo_poblacion_trimestre = pd.DataFrame(filas_poblacion)
riesgo_poblacion_trimestre.to_csv(META / f"riesgo_poblacion_trimestre{SUFIJO}.csv", index=False)

resumen_poblacion = riesgo_poblacion_trimestre.groupby("era")[
    ["n_hogares_total"] + [f"n_precario_{d}" for d in DEFINICIONES]
].mean().round(0)
for d in DEFINICIONES:
    resumen_poblacion[f"pct_{d}"] = (riesgo_poblacion_trimestre.groupby("era").apply(
        lambda g, d=d: (g[f"n_precario_{d}"].sum() / g["n_hogares_total"].sum() * 100), include_groups=False)).round(1)
print("--- N promedio por trimestre y % sobre el total de hogares, por era y definición (tabla a mostrar) ---")
display(resumen_poblacion)

## Sección 2 — Etiqueta de salida `t+h` (`h=1,4`), por definición

Sobre `pares_h{h}` con `identidad_confirmada==True` (`h=4` excluye
`ORIGENES_EXCLUIDOS_D13`). La categoría de salida **depende de qué abandona cada
definición** -- no es una regla única (corrección de Iván):

- **a** (Q1-Q2 + activos sin empleo formal) -- `ingreso_alto_th = decil_th>=5` (Q3+):
  `sale_por_empleo` (prioridad máxima, formal en `t+h`) > `sale_por_ingreso_sin_activos`
  (sube a Q3+ **sin** nadie activo -- ingresos no laborales, se cuenta aparte) >
  `sale_por_ingreso_con_activos` > `pasa_a_sin_activos` (sin ingreso alto, sin activos) >
  `permanece`.
- **b** (activos sin empleo formal, sin condición de ingreso) -- no hay ingreso que
  abandonar: `sale_por_empleo` > `pasa_a_sin_activos` > `permanece`.
- **c** (Q1 solo) -- la precariedad de c no depende de actividad, así que acá **no
  existe** `pasa_a_sin_activos` como categoría aparte: `sale` = `decil_th>=3` (Q2+), sin
  importar actividad; el resto es `permanece` (= sigue en Q1). `con_empleo_formal_
  adicional` (entre los `sale`) y `sin_activos_en_q1` (entre los `permanece`) son
  columnas **descriptivas**, no cambian la categoría.

`sale_binario` tiene dos variantes, siempre reportadas las dos: **principal**
(`pasa_a_sin_activos` cuenta como "no sale" -- excluirla sesga la tasa hacia arriba) y
**robustez** (`pasa_a_sin_activos` se excluye del denominador). Para **c** las dos son
idénticas (no hay `pasa_a_sin_activos`) -- se reporta una sola vez. `no_determinable` es
siempre `NA` en ambas variantes.

In [ ]:
def categorizar_a(decil_th: pd.Series, algun_formal_th: pd.Series, algun_activo_th: pd.Series) -> pd.Series:
    ingreso_alto = (decil_th >= 5).astype("boolean")
    sin_activos = (algun_activo_th == False).astype("boolean")
    cat = pd.Series("permanece", index=decil_th.index, dtype="object")
    cat[sin_activos.fillna(False) & ~ingreso_alto.fillna(False)] = "pasa_a_sin_activos"
    cat[ingreso_alto.fillna(False) & sin_activos.fillna(False)] = "sale_por_ingreso_sin_activos"
    cat[ingreso_alto.fillna(False) & ~sin_activos.fillna(True)] = "sale_por_ingreso_con_activos"
    indeterminado = algun_formal_th.isna() | ingreso_alto.isna() | algun_activo_th.isna()
    cat[indeterminado] = "no_determinable"
    cat[algun_formal_th.fillna(False) == True] = "sale_por_empleo"  # máxima prioridad, pisa todo lo anterior
    return cat


def categorizar_b(algun_formal_th: pd.Series, algun_activo_th: pd.Series) -> pd.Series:
    cat = pd.Series("permanece", index=algun_formal_th.index, dtype="object")
    cat[(algun_activo_th == False).fillna(False)] = "pasa_a_sin_activos"
    indeterminado = algun_formal_th.isna() | algun_activo_th.isna()
    cat[indeterminado] = "no_determinable"
    cat[algun_formal_th.fillna(False) == True] = "sale_por_empleo"
    return cat


def categorizar_c(decil_th: pd.Series, algun_formal_th: pd.Series, algun_activo_th: pd.Series) -> tuple[pd.Series, pd.Series, pd.Series]:
    sale = (decil_th >= 3).astype("boolean")
    cat = pd.Series("permanece", index=decil_th.index, dtype="object")
    cat[sale.fillna(False)] = "sale"
    cat[sale.isna()] = "no_determinable"
    con_empleo_adicional = pd.Series(pd.NA, index=decil_th.index, dtype="boolean")
    con_empleo_adicional[cat == "sale"] = algun_formal_th[cat == "sale"]
    sin_activos_en_q1 = pd.Series(pd.NA, index=decil_th.index, dtype="boolean")
    sin_activos_en_q1[cat == "permanece"] = (algun_activo_th == False)[cat == "permanece"]
    return cat, con_empleo_adicional, sin_activos_en_q1


CATEGORIAS_SALE = {
    "a": {"sale_por_empleo", "sale_por_ingreso_con_activos", "sale_por_ingreso_sin_activos"},
    "b": {"sale_por_empleo"},
    "c": {"sale"},
}


def calcular_sale_binario(categoria: pd.Series, definicion: str, variante: str) -> pd.Series:
    resultado = pd.Series(pd.NA, index=categoria.index, dtype="boolean")
    resultado[categoria.isin(CATEGORIAS_SALE[definicion])] = True
    resultado[categoria == "permanece"] = False
    if definicion != "c" and variante == "principal":
        resultado[categoria == "pasa_a_sin_activos"] = False
    # robustez (y "c", donde no aplica pasa_a_sin_activos): queda NA, ya es el default.
    return resultado


def categorizar_resultado(definicion: str, decil_th: pd.Series, algun_formal_th: pd.Series,
                           algun_activo_th: pd.Series) -> pd.Series:
    if definicion == "a":
        return categorizar_a(decil_th, algun_formal_th, algun_activo_th)
    if definicion == "b":
        return categorizar_b(algun_formal_th, algun_activo_th)
    cat, _, _ = categorizar_c(decil_th, algun_formal_th, algun_activo_th)
    return cat

In [ ]:
def construir_pares_precariedad(h: int, anio_t: int, trimestre_t: int) -> pd.DataFrame | None:
    p_pares = pm.ruta_particion(PANEL / f"pares_h{h}", "", anio_t, trimestre_t)
    if not p_pares.exists():
        return None
    pares = pd.read_parquet(p_pares, columns=["CODUSU", "AGLOMERADO", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada"])
    pares = pares[pares["identidad_confirmada"] == True]
    if pares.empty:
        return None
    anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
    prec_t, prec_th = CACHE_PRECARIEDAD.get((anio_t, trimestre_t)), CACHE_PRECARIEDAD.get((anio_th, trimestre_th))
    if prec_t is None or prec_th is None:
        return None
    prec_t, prec_th = prec_t.reset_index(), prec_th.reset_index()
    cols_t = {c: f"{c}_t" for c in prec_t.columns if c not in ("CODUSU", "NRO_HOGAR")}
    cols_th = {c: f"{c}_th" for c in prec_th.columns if c not in ("CODUSU", "NRO_HOGAR")}
    base = pares.rename(columns={"NRO_HOGAR_t": "NRO_HOGAR"}).merge(
        prec_t.rename(columns=cols_t), on=["CODUSU", "NRO_HOGAR"], how="left"
    ).rename(columns={"NRO_HOGAR": "NRO_HOGAR_t"})
    base = base.merge(
        prec_th.rename(columns={"NRO_HOGAR": "NRO_HOGAR_th", **cols_th}), on=["CODUSU", "NRO_HOGAR_th"], how="left"
    )
    base["cluster_id"] = base["CODUSU"].astype(str) + "_" + base["AGLOMERADO"].astype(str)
    return base


COLUMNAS_CATEGORIA = {
    "a": ["sale_por_empleo", "sale_por_ingreso_con_activos", "sale_por_ingreso_sin_activos", "pasa_a_sin_activos", "permanece", "no_determinable"],
    "b": ["sale_por_empleo", "pasa_a_sin_activos", "permanece", "no_determinable"],
    "c": ["sale", "permanece", "no_determinable"],
}

filas_salida = []
filas_pares_riesgo = []
for h in (1, 4):
    for anio_t, trimestre_t in trimestres_a_procesar:
        if h == 4 and (anio_t, trimestre_t) in ORIGENES_EXCLUIDOS_D13:
            continue
        base = construir_pares_precariedad(h, anio_t, trimestre_t)
        if base is None:
            continue
        era = "historica" if es_historico(anio_t, trimestre_t) else "regular"
        for d in DEFINICIONES:
            en_riesgo = base[f"precario_{d}_t"] == True
            if not en_riesgo.any():
                continue
            sub = base.loc[en_riesgo]
            if d == "a":
                categoria = categorizar_a(sub["decil_th"], sub["algun_formal_th"], sub["algun_activo_th"])
            elif d == "b":
                categoria = categorizar_b(sub["algun_formal_th"], sub["algun_activo_th"])
            else:
                categoria, con_empleo_adic, _ = categorizar_c(sub["decil_th"], sub["algun_formal_th"], sub["algun_activo_th"])

            conteo = categoria.value_counts()
            fila = {"h": h, "anio_t": anio_t, "trimestre_t": trimestre_t, "era": era, "definicion": d,
                    "n_precario_t": len(sub)}
            for col in COLUMNAS_CATEGORIA[d]:
                fila[f"n_{col}"] = int(conteo.get(col, 0))
            if d == "c":
                fila["n_sale_con_empleo_formal_adicional"] = int((con_empleo_adic == True).sum())
            else:
                fila["n_sale_con_empleo_formal_adicional"] = fila.get("n_sale_por_empleo", 0)

            sale_principal = calcular_sale_binario(categoria, d, "principal")
            sale_robustez = sale_principal if d == "c" else calcular_sale_binario(categoria, d, "robustez")
            for nombre, serie in (("principal", sale_principal), ("robustez", sale_robustez)):
                n_det = serie.notna().sum()
                fila[f"tasa_salida_{nombre}"] = round((serie == True).sum() / n_det * 100, 2) if n_det else np.nan
            filas_salida.append(fila)

            filas_pares_riesgo.append(pd.DataFrame({
                "h": h, "definicion": d, "anio_t": anio_t, "trimestre_t": trimestre_t, "era": era,
                "cluster_id": sub["cluster_id"].to_numpy(), "decil_t": sub["decil_t"].to_numpy(),
                "ancla_con_deficit_t": sub["ancla_con_deficit_t"].astype("boolean").to_numpy(),
                "categoria": categoria.to_numpy(),
                "sale_binario_principal": sale_principal.to_numpy(), "sale_binario_robustez": sale_robustez.to_numpy(),
            }))

riesgo_salida_trimestre = pd.DataFrame(filas_salida)
riesgo_salida_trimestre.to_csv(META / f"riesgo_salida_trimestre{SUFIJO}.csv", index=False)
pares_riesgo = pd.concat(filas_pares_riesgo, ignore_index=True) if filas_pares_riesgo else pd.DataFrame()
print(f"{len(riesgo_salida_trimestre):,} filas trimestre x era x definición x h, {len(pares_riesgo):,} pares en riesgo (fila a fila)")

print("--- tasa de salida pooled por era, h y definición -- principal / robustez (tabla a mostrar) ---")
def _pooled(g):
    return pd.Series({
        "n_precario_t_total": len(g),
        "tasa_principal_pooled": round((g["sale_binario_principal"] == True).sum() / max(g["sale_binario_principal"].notna().sum(), 1) * 100, 1),
        "tasa_robustez_pooled": round((g["sale_binario_robustez"] == True).sum() / max(g["sale_binario_robustez"].notna().sum(), 1) * 100, 1),
    })

display(pares_riesgo.groupby(["h", "era", "definicion"]).apply(_pooled, include_groups=False))

## Sección 2b — Salida según `ancla_con_deficit` en `t`, con IC bootstrap (corregido)

Usa `diferencia_dos_grupos_bootstrap_cluster` de `src/bootstrap.py` (import, no
duplicado -- ver celda 0) sobre `sale_binario_principal`. La versión anterior tenía un
bug que daba un IC ~30x más angosto de lo real -- corregido y testeado en
`tests/test_bootstrap.py`.

In [ ]:
filas_bootstrap = []
if not pares_riesgo.empty:
    for h in (1, 4):
        for d in DEFINICIONES:
            sub = pares_riesgo[(pares_riesgo["h"] == h) & (pares_riesgo["definicion"] == d)]
            if sub.empty:
                continue
            resultado = diferencia_dos_grupos_bootstrap_cluster(
                sub, "cluster_id", "sale_binario_principal", "ancla_con_deficit_t", True, False)
            filas_bootstrap.append({
                "h": h, "definicion": d,
                "tasa_con_deficit": resultado["tasa_a"], "tasa_sin_deficit": resultado["tasa_b"],
                "diferencia": resultado["diferencia"],
                "diferencia_ic95_low": resultado["diferencia_ic95_low"], "diferencia_ic95_high": resultado["diferencia_ic95_high"],
                "n_con_deficit": resultado["n_a"], "n_sin_deficit": resultado["n_b"], "n_clusters": resultado["n_clusters"],
            })

riesgo_salida_ancla_bootstrap = pd.DataFrame(filas_bootstrap)
riesgo_salida_ancla_bootstrap.to_csv(META / f"riesgo_salida_ancla_bootstrap{SUFIJO}.csv", index=False)
print("--- tasa de salida (principal) según ancla_con_deficit en t, IC95% bootstrap corregido (tabla a mostrar) ---")
display(riesgo_salida_ancla_bootstrap)

## `n_no_determinable` por era y por `ancla_con_deficit_t`

En la era regular, `decil_th` falta cuando el hogar no declara ingreso en `t+h` (D3) --
hace falta ver si esa falta de dato es pareja entre quienes tenían déficit habitacional
en `t` y quienes no (si no lo fuera, el IC de la sección 2b subestimaría un problema de
missingness diferencial).

In [ ]:
filas_no_det = []
if not pares_riesgo.empty:
    grp = pares_riesgo.groupby(["definicion", "h", "era", "ancla_con_deficit_t"])
    for (d, h, era, ancla), sub in grp:
        n_total = len(sub)
        n_no_det = int((sub["categoria"] == "no_determinable").sum())
        filas_no_det.append({
            "definicion": d, "h": h, "era": era, "ancla_con_deficit_t": bool(ancla) if pd.notna(ancla) else None,
            "n_total": n_total, "n_no_determinable": n_no_det,
            "pct_no_determinable": round(n_no_det / n_total * 100, 1) if n_total else np.nan,
        })

riesgo_no_determinable_por_ancla = pd.DataFrame(filas_no_det)
riesgo_no_determinable_por_ancla.to_csv(META / f"riesgo_no_determinable_por_ancla{SUFIJO}.csv", index=False)
print("--- % no_determinable por era y ancla_con_deficit_t, era regular (tabla a mostrar) ---")
display(riesgo_no_determinable_por_ancla[riesgo_no_determinable_por_ancla["era"] == "regular"]
        .pivot_table(index=["definicion", "h"], columns="ancla_con_deficit_t", values="pct_no_determinable"))

## Sección 3 — Profundidad: la brecha por `ancla_con_deficit` dentro de cada decil

¿La diferencia de salida por déficit habitacional (sección 2b) es un efecto de
composición (los hogares con déficit están más concentrados en los deciles más bajos,
que de por sí salen menos) o sobrevive **dentro** de cada decil? Dos lecturas:
- **Por decil simple**: la misma comparación de la 2b, filtrada a cada decil de `t`.
- **Estandarizada**: `estandarizar_diferencia_bootstrap` (puerto de
  `resumen_estandarizado` del notebook 03, celda 37, parametrizado por un solo estrato en
  vez de quintil×educación) -- tasa observada de "con déficit" vs. la que tendría si
  cada decil tuviera la tasa de "sin déficit" de ese mismo decil, ponderado por la propia
  distribución de deciles de "con déficit".

In [ ]:
filas_por_decil = []
filas_estandarizada = []
if not pares_riesgo.empty:
    for h in (1, 4):
        for d in DEFINICIONES:
            sub = pares_riesgo[(pares_riesgo["h"] == h) & (pares_riesgo["definicion"] == d) & pares_riesgo["decil_t"].notna()]
            if sub.empty:
                continue
            for decil, sub_decil in sub.groupby("decil_t"):
                resultado = diferencia_dos_grupos_bootstrap_cluster(
                    sub_decil, "cluster_id", "sale_binario_principal", "ancla_con_deficit_t", True, False, n_boot=400)
                filas_por_decil.append({
                    "definicion": d, "h": h, "decil": int(decil),
                    "tasa_con_deficit": resultado["tasa_a"], "tasa_sin_deficit": resultado["tasa_b"],
                    "diferencia": resultado["diferencia"],
                    "diferencia_ic95_low": resultado["diferencia_ic95_low"], "diferencia_ic95_high": resultado["diferencia_ic95_high"],
                    "n_con_deficit": resultado["n_a"], "n_sin_deficit": resultado["n_b"], "n_clusters": resultado["n_clusters"],
                })

            resultado_std = estandarizar_diferencia_bootstrap(
                sub, "cluster_id", "sale_binario_principal", "ancla_con_deficit_t", True, False, "decil_t", n_boot=400)
            filas_estandarizada.append({
                "definicion": d, "h": h,
                "pct_observado_con_deficit": resultado_std["pct_observado_con"],
                "pct_estandarizado_sin_deficit": resultado_std["pct_estandarizado_sin"],
                "diferencia": resultado_std["diferencia"],
                "diferencia_ic95_low": resultado_std["diferencia_ic95_low"], "diferencia_ic95_high": resultado_std["diferencia_ic95_high"],
                "n_clusters": resultado_std["n_clusters"], "n_con_deficit": resultado_std["n_con"],
            })

riesgo_salida_ancla_por_decil = pd.DataFrame(filas_por_decil)
riesgo_salida_ancla_por_decil.to_csv(META / f"riesgo_salida_ancla_por_decil{SUFIJO}.csv", index=False)
riesgo_salida_ancla_estandarizada = pd.DataFrame(filas_estandarizada)
riesgo_salida_ancla_estandarizada.to_csv(META / f"riesgo_salida_ancla_estandarizada{SUFIJO}.csv", index=False)

print("--- diferencia por decil, h=1 (tabla a mostrar) ---")
display(riesgo_salida_ancla_por_decil[riesgo_salida_ancla_por_decil["h"] == 1]
        .pivot_table(index="decil", columns="definicion", values="diferencia"))
print("--- diferencia estandarizada por decil vs. la pooled de la sección 2b (tabla a mostrar) ---")
display(riesgo_salida_ancla_estandarizada)

## Sección 2c — Variante estricta (con trayectorias del notebook 04)

"Sale y se mantiene fuera en la visita siguiente", usando la visita siguiente real del
hogar en la rotación 2-2-2 (`t0+4` para `h=1`, `t0+5` para `h=4`). Misma categorización
por definición que la sección 2, aplicada en los 2 puntos de verificación; `estricta =
sale_binario(punto) Y sale_binario(siguiente)` -- el `&` de pandas con dtype `"boolean"`
sigue lógica de Kleene, que acá da exactamente lo que hace falta (si el punto siguiente
muestra `False` ya alcanza para decidir `estricta=False`, conozca o no el primer punto
con certeza). Se reportan las 4 combinaciones (simple/estricta × principal/robustez);
para **c**, robustez==principal (no se duplica).

In [ ]:
def cargar_trayectoria(anio_t0: int, trimestre_t0: int) -> pd.DataFrame | None:
    p = pm.ruta_particion(TRAYECTORIAS / "trayectorias", "", anio_t0, trimestre_t0)
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    return df[df["identidad_confirmada_trayectoria"] == True]


COLUMNAS_PRECARIEDAD_TRAYECTORIA = ["decil", "algun_formal", "algun_activo"] + [f"precario_{d}" for d in DEFINICIONES]


def unir_precariedad(df: pd.DataFrame, anio: int, trimestre: int, sufijo: str) -> pd.DataFrame:
    prec = CACHE_PRECARIEDAD.get((anio, trimestre))
    if prec is None:
        for c in COLUMNAS_PRECARIEDAD_TRAYECTORIA:
            df[f"{c}{sufijo}"] = pd.NA
        return df
    prec = prec.reset_index()[["CODUSU", "NRO_HOGAR"] + COLUMNAS_PRECARIEDAD_TRAYECTORIA]
    prec = prec.rename(columns={c: f"{c}{sufijo}" for c in COLUMNAS_PRECARIEDAD_TRAYECTORIA})
    return df.merge(prec, on=["CODUSU", "NRO_HOGAR"], how="left")


filas_estricta = []
for anio_t0, trimestre_t0 in trimestres_a_procesar:
    if (anio_t0, trimestre_t0) in ORIGENES_EXCLUIDOS_D13:
        continue
    tray = cargar_trayectoria(anio_t0, trimestre_t0)
    if tray is None or tray.empty:
        continue
    era = "historica" if es_historico(anio_t0, trimestre_t0) else "regular"
    anio_t1, trimestre_t1 = panel.sumar_trimestres(anio_t0, trimestre_t0, 1)
    anio_t4, trimestre_t4 = panel.sumar_trimestres(anio_t0, trimestre_t0, 4)
    anio_t5, trimestre_t5 = panel.sumar_trimestres(anio_t0, trimestre_t0, 5)

    base = tray[["CODUSU", "NRO_HOGAR", "AGLOMERADO"]].copy()
    base = unir_precariedad(base, anio_t0, trimestre_t0, "_t0")
    base = unir_precariedad(base, anio_t1, trimestre_t1, "_t0_1")
    base = unir_precariedad(base, anio_t4, trimestre_t4, "_t0_4")
    base = unir_precariedad(base, anio_t5, trimestre_t5, "_t0_5")

    for d in DEFINICIONES:
        en_riesgo = base[f"precario_{d}_t0"] == True
        if not en_riesgo.any():
            continue
        sub = base.loc[en_riesgo]

        for h, suf_punto, suf_siguiente, nombre_siguiente in ((1, "_t0_1", "_t0_4", "t0+4"), (4, "_t0_4", "_t0_5", "t0+5")):
            cat_punto = categorizar_resultado(d, sub[f"decil{suf_punto}"], sub[f"algun_formal{suf_punto}"], sub[f"algun_activo{suf_punto}"])
            cat_siguiente = categorizar_resultado(d, sub[f"decil{suf_siguiente}"], sub[f"algun_formal{suf_siguiente}"], sub[f"algun_activo{suf_siguiente}"])

            sale_p_principal = calcular_sale_binario(cat_punto, d, "principal")
            sale_s_principal = calcular_sale_binario(cat_siguiente, d, "principal")
            if d == "c":
                sale_p_robustez, sale_s_robustez = sale_p_principal, sale_s_principal
            else:
                sale_p_robustez = calcular_sale_binario(cat_punto, d, "robustez")
                sale_s_robustez = calcular_sale_binario(cat_siguiente, d, "robustez")

            variantes = {
                "simple_principal": sale_p_principal, "simple_robustez": sale_p_robustez,
                "estricta_principal": sale_p_principal & sale_s_principal,
                "estricta_robustez": sale_p_robustez & sale_s_robustez,
            }
            fila = {"anio_t0": anio_t0, "trimestre_t0": trimestre_t0, "era": era, "definicion": d, "h": h,
                    "visita_siguiente": nombre_siguiente, "n_precario_t0": len(sub)}
            for nombre, serie in variantes.items():
                n_det = int(serie.notna().sum())
                fila[f"tasa_{nombre}"] = round((serie == True).sum() / n_det * 100, 1) if n_det else np.nan
                fila[f"n_sale_{nombre}"] = int((serie == True).sum())
                fila[f"n_determinable_{nombre}"] = n_det
            filas_estricta.append(fila)

riesgo_salida_estricta = pd.DataFrame(filas_estricta)
riesgo_salida_estricta.to_csv(META / f"riesgo_salida_estricta{SUFIJO}.csv", index=False)
print(f"{len(riesgo_salida_estricta):,} filas cohorte x definición x h")

print("--- tasa simple vs. estricta, principal, pooled por era/h/definición (tabla a mostrar) ---")
def _pooled_estricta(g):
    return pd.Series({
        "tasa_simple_principal_pooled": round(g["n_sale_simple_principal"].sum() / max(g["n_determinable_simple_principal"].sum(), 1) * 100, 1),
        "tasa_estricta_principal_pooled": round(g["n_sale_estricta_principal"].sum() / max(g["n_determinable_estricta_principal"].sum(), 1) * 100, 1),
        "n_precario_t0_total": int(g["n_precario_t0"].sum()),
    })
display(riesgo_salida_estricta.groupby(["h", "era", "definicion"]).apply(_pooled_estricta, include_groups=False))

## Sección 4 — Exclusión de la población en riesgo por D3

Las definiciones **a**/**c** usan `DECCFR`: un hogar sin quintil válido (`decil_0` +
`ingreso_no_declarado`) no puede clasificarse y queda fuera de la población en riesgo de
esas dos definiciones (no de **b**). Se cuantifica cuántos son, por trimestre/era, y si
su perfil de vivienda (`ancla_con_deficit`) y empleo (`activos_sin_empleo_formal`)
difiere de los hogares que sí tienen quintil.

In [ ]:
filas_exclusion = []
for (anio, trimestre) in trimestres_a_procesar:
    prec = CACHE_PRECARIEDAD.get((anio, trimestre))
    if prec is None:
        continue
    era = "historica" if es_historico(anio, trimestre) else "regular"
    con_quintil = prec["grupo_ingreso"].isin(ORDEN_QUINTILES)
    sin_quintil = ~con_quintil & prec["grupo_ingreso"].notna()
    for grupo, mask in (("con_quintil", con_quintil), ("sin_quintil", sin_quintil)):
        sub = prec[mask]
        if sub.empty:
            continue
        filas_exclusion.append({
            "anio": anio, "trimestre": trimestre, "era": era, "grupo": grupo, "n_hogares": len(sub),
            "pct_ancla_con_deficit": round((sub["ancla_con_deficit"] == True).mean() * 100, 1),
            "pct_activos_sin_empleo_formal": round((sub["activos_sin_empleo_formal"] == True).mean() * 100, 1),
        })

riesgo_exclusion_d3 = pd.DataFrame(filas_exclusion)
riesgo_exclusion_d3.to_csv(META / f"riesgo_exclusion_d3{SUFIJO}.csv", index=False)
print("--- perfil con/sin quintil, promedio por era (tabla a mostrar) ---")
display(riesgo_exclusion_d3[riesgo_exclusion_d3["era"] == "regular"].groupby("grupo")[
    ["n_hogares", "pct_ancla_con_deficit", "pct_activos_sin_empleo_formal"]].mean().round(1))

## Sección 5 — Inventario de predictores disponibles en `t`

Todas las columnas de `01_armonizado/{hogar,individual}`, clasificadas por bloque con un
mapeo por prefijo -- automático, no verificado variable por variable contra el codebook.

**Corrección de encuadre (Iván)**: la fuga de datos es cualquier variable medida en
`t+h` usada para predecir el resultado en `t+h` -- **no** la posición de ingreso en `t`.
`DECCFR_t` (y toda la familia de deciles de `t`) sigue siendo insumo de la población en
riesgo y **candidato legítimo a predictor** -- va a un bloque propio
`ingresos_posicion_t`, separado de `identificadores_pesos_excluidos`. Lo que nunca puede
entrar como predictor es `DECCFR_th` ni ninguna otra variable de `t+h` -- eso sí sería
fuga, por definición (la etiqueta se construye a partir de variables de `t+h`).
`PONDERA`/`PONDIH` y el resto de identificadores/pesos/campos auxiliares siguen
excluidos (no son predictores, son metadata).

In [ ]:
import re

PREFIJOS_EXCLUIDOS = (
    "CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE", "PONDERA", "PONDIH", "PONDII", "PONDIIO",
    "REALIZADA", "IMPUTA", "CH05", "IDIMPH", "IDIMPP", "FILTER__",
)
SUFIJOS_EXCLUIDOS = ("_ESP", "_COD", "_ANO", "_MES", "_DIA")
# La familia de deciles aparece también truncada a 8 caracteres en algunos trimestres
# históricos (ej. "P_DECCF" en vez de "P_DECCFR").
PATRON_DECIL = re.compile(r"^(P_)?[AGIPR]?DEC(CFR?|IFR?|INDR?|OCUR?)$")


def clasificar_variable(var: str) -> str:
    if var in PREFIJOS_EXCLUIDOS or var.endswith(SUFIJOS_EXCLUIDOS) or var.upper().startswith("UNNAMED"):
        return "identificadores_pesos_excluidos"
    if PATRON_DECIL.match(var):
        return "ingresos_posicion_t"  # candidato a predictor en t, NO es fuga (ver celda anterior)
    if var.startswith("II") or var.startswith("IV"):
        return "vivienda"
    if var.startswith("VII"):
        return "sin_clasificar"
    if var in ("CH03", "CH04", "CH06", "CH07") or var.startswith("IX_"):
        return "composicion"
    if var == "NIVEL_ED" or var == "CH09" or re.match(r"^CH1[0-6]", var):
        return "educacion"
    if var in ("ESTADO", "CAT_OCUP", "CAT_INAC", "EMPLEO", "SECTOR", "INTENSI") or re.match(r"^PP\d", var):
        return "trabajo"
    if re.match(r"^V\d", var) or var in ("IPCF", "ITF", "P21", "T_VI", "P47T"):
        return "ingresos_por_fuente"
    if var in ("AGLOMERADO", "REGION", "MAS_500"):
        return "aglomerado_contexto"
    return "sin_clasificar"


COLUMNAS_PARES_CAMBIO = ["cambio_composicion", "cambio_jefatura", "n_altas", "n_bajas", "diferencia_edad_jefe"]

inventario = pd.read_csv(META / "inventario_variables.csv")
inventario["era"] = np.where(inventario["anio"] <= 2015, "historica", "regular")
existencia = (inventario.groupby(["tipo", "variable", "era"]).size().unstack(fill_value=0) > 0)
existencia.columns = [f"existe_{c}" for c in existencia.columns]
existencia = existencia.reset_index()

MUESTRA_FALTANTES = {"historica": [(2004, 2), (2010, 2), (2015, 2)], "regular": [(2017, 2), (2021, 2), (2024, 2)]}


def pct_faltantes_muestra(tipo: str, trimestres: list[tuple[int, int]]) -> pd.Series:
    partes = []
    for anio, trimestre in trimestres:
        p = pm.ruta_particion(ARMONIZADO, tipo, anio, trimestre)
        if p.exists():
            partes.append(pd.read_parquet(p))
    if not partes:
        return pd.Series(dtype="float64")
    df = pd.concat(partes, ignore_index=True)
    return df.isna().mean() * 100


filas_inventario = []
for tipo in ("hogar", "individual"):
    faltantes_era = {era: pct_faltantes_muestra(tipo, trims) for era, trims in MUESTRA_FALTANTES.items()}
    columnas = sorted(set(existencia.loc[existencia["tipo"] == tipo, "variable"]))
    for var in columnas:
        bloque = clasificar_variable(var)
        fila = {"tipo": tipo, "variable": var, "bloque": bloque}
        exist_row = existencia[(existencia["tipo"] == tipo) & (existencia["variable"] == var)]
        fila["existe_historica"] = bool(exist_row["existe_historica"].iloc[0]) if "existe_historica" in exist_row.columns and len(exist_row) else False
        fila["existe_regular"] = bool(exist_row["existe_regular"].iloc[0]) if "existe_regular" in exist_row.columns and len(exist_row) else False
        for era, serie in faltantes_era.items():
            fila[f"pct_faltante_{era}"] = round(float(serie.get(var, np.nan)), 1) if var in serie.index else np.nan
        filas_inventario.append(fila)

riesgo_predictores_inventario = pd.DataFrame(filas_inventario)
riesgo_predictores_inventario.to_csv(META / f"riesgo_predictores_inventario{SUFIJO}.csv", index=False)

print(f"{len(riesgo_predictores_inventario)} variables inventariadas")
print("--- cantidad de variables por bloque y tipo (tabla a mostrar) ---")
display(riesgo_predictores_inventario.groupby(["bloque", "tipo"]).size().unstack(fill_value=0))
print()
print("--- bloque 'cambio respecto de la visita anterior' (solo hogares vinculados, pares_h1/h4) ---")
print(COLUMNAS_PARES_CAMBIO)

## D14 (borrador) — Definición de "hogar en riesgo", PENDIENTE de confirmación de Iván

Generado desde los números de las secciones 1 a 5 -- no elige ninguna de las 3
candidatas, las deja con datos reales para que la decisión la tome Iván. Reescrito tras
las 4 correcciones pedidas en la revisión.

In [ ]:
from IPython.display import Markdown, display as _display


def _tasa_pooled(d, h, era, variante):
    pp = pares_riesgo[(pares_riesgo["h"] == h) & (pares_riesgo["definicion"] == d) & (pares_riesgo["era"] == era)]
    col = f"sale_binario_{variante}"
    det = pp[col].notna().sum()
    return round((pp[col] == True).sum() / det * 100, 1) if det else float("nan")


pct_poblacion = riesgo_poblacion_trimestre.groupby("era").apply(
    lambda g: pd.Series({d: round(g[f"n_precario_{d}"].sum() / g["n_hogares_total"].sum() * 100, 1) for d in DEFINICIONES}),
    include_groups=False)

filas_md = ["| definición | % en riesgo (hist./reg.) | salida h=1 principal (hist./reg.) | "
            "salida h=1 robustez (hist./reg.) | salida h=4 principal (hist./reg.) | dif. por ancla h=1, IC95% |",
            "|---|---|---|---|---|---|"]
for d in DEFINICIONES:
    pob_h = pct_poblacion.loc["historica", d] if "historica" in pct_poblacion.index else float("nan")
    pob_r = pct_poblacion.loc["regular", d] if "regular" in pct_poblacion.index else float("nan")

    boot = riesgo_salida_ancla_bootstrap[(riesgo_salida_ancla_bootstrap["definicion"] == d) & (riesgo_salida_ancla_bootstrap["h"] == 1)]
    if len(boot):
        b = boot.iloc[0]
        dif_txt = f"{b['diferencia']:+.1f}pp [{b['diferencia_ic95_low']:+.1f}, {b['diferencia_ic95_high']:+.1f}]"
    else:
        dif_txt = "s/d"

    robustez_txt = (f"{_tasa_pooled(d,1,'historica','robustez'):.1f}% / {_tasa_pooled(d,1,'regular','robustez'):.1f}%"
                     if d != "c" else "= principal (c no tiene pasa_a_sin_activos)")

    filas_md.append(
        f"| **{d}** ({DESCRIPCION_DEFINICION[d]}) | {pob_h:.1f}% / {pob_r:.1f}% | "
        f"{_tasa_pooled(d,1,'historica','principal'):.1f}% / {_tasa_pooled(d,1,'regular','principal'):.1f}% | "
        f"{robustez_txt} | "
        f"{_tasa_pooled(d,4,'historica','principal'):.1f}% / {_tasa_pooled(d,4,'regular','principal'):.1f}% | {dif_txt} |"
    )

# Cuánto sobrevive la diferencia dentro de decil (estandarizada) vs. la pooled (h=1)
filas_profundidad = ["| definición | diferencia pooled (sección 2b) | diferencia estandarizada por decil | "
                      "% que sobrevive dentro de decil |", "|---|---|---|---|"]
for d in DEFINICIONES:
    pooled = riesgo_salida_ancla_bootstrap[(riesgo_salida_ancla_bootstrap["definicion"] == d) & (riesgo_salida_ancla_bootstrap["h"] == 1)]
    std = riesgo_salida_ancla_estandarizada[(riesgo_salida_ancla_estandarizada["definicion"] == d) & (riesgo_salida_ancla_estandarizada["h"] == 1)]
    if len(pooled) and len(std):
        dif_pooled, dif_std = pooled.iloc[0]["diferencia"], std.iloc[0]["diferencia"]
        pct_sobrevive = round(dif_std / dif_pooled * 100, 0) if dif_pooled else float("nan")
        filas_profundidad.append(f"| **{d}** | {dif_pooled:+.1f}pp | {dif_std:+.1f}pp | {pct_sobrevive:.0f}% |")

# n_no_determinable, era regular: ¿difiere por ancla?
tabla_no_det_regular = riesgo_no_determinable_por_ancla[
    (riesgo_no_determinable_por_ancla["era"] == "regular") & (riesgo_no_determinable_por_ancla["h"] == 1)
].pivot_table(index="definicion", columns="ancla_con_deficit_t", values="pct_no_determinable")


def _tabla_a_md(df, nombre_indice):
    lineas = [f"| {nombre_indice} | " + " | ".join(str(c) for c in df.columns) + " |",
              "|" + "---|" * (len(df.columns) + 1)]
    for idx, fila in df.iterrows():
        lineas.append(f"| {idx} | " + " | ".join(f"{v:.1f}" if pd.notna(v) else "s/d" for v in fila) + " |")
    return "\n".join(lineas)

texto_d14 = f'''## D14 (borrador, generado) — Definición de "hogar en riesgo", **PENDIENTE de confirmación de Iván**

Tres candidatas evaluadas en paralelo, ninguna elegida -- números reales de
`notebooks/06_poblacion_riesgo.ipynb`, corridos sobre {len(trimestres_a_procesar)} trimestres de origen.
Reescrito tras 4 correcciones de Iván sobre la primera versión (bug de bootstrap, categorías de
salida por definición, profundidad por decil, encuadre del inventario).

### Población y tasa de salida

''' + "\n".join(filas_md) + '''

`tasa_salida_principal` cuenta pasar a no tener ningún miembro activo (`pasa_a_sin_activos`)
como "no sale" -- es la lectura recomendada por default (la versión que lo excluye del
denominador, `robustez`, sesga la tasa hacia arriba al condicionar en el resultado). Para
**c** no existe esa distinción: la precariedad de c no depende de actividad, así que
`robustez == principal` siempre.

### Profundidad: ¿la brecha por déficit habitacional es composición de decil o sobrevive dentro de cada uno?

''' + "\n".join(filas_profundidad) + '''

### `n_no_determinable` por `ancla_con_deficit_t`, era regular, h=1 (% sobre la población en riesgo de cada definición)

''' + _tabla_a_md(tabla_no_det_regular, "definición") + '''

(`decil_th` falta cuando el hogar no declara ingreso en `t+h`, D3 -- si este % difiere
mucho entre con/sin déficit, el IC de la sección 2b podría estar subestimando un problema
de missingness diferencial; se deja el número, sin forzar una lectura.)

### No elegida -- puntos para la decisión de Iván

- (a)/(c) dependen de `DECCFR`: quedan sin población en riesgo clasificable los hogares
  de la era regular que no declaran ingreso (sección 4, D3).
- (b) no depende de ingreso -- población en riesgo más estable entre eras, pero no
  distingue un hogar realmente precario de un independiente informal de ingresos altos.
- `sale_por_ingreso_sin_activos` (definición a) -- subir de decil sin nadie activo
  (ingresos no laborales) es una salida real pero de mecanismo distinto a conseguir
  trabajo; se reporta aparte en `riesgo_salida_trimestre.csv`.
- La variante estricta con trayectorias (sección 2c) da tasas más bajas que la simple en
  las 3 -- una parte de las "salidas" simples son inestables.
- El inventario de predictores (sección 5) ya no excluye la familia de deciles de `t`
  como si fuera fuga -- la fuga es cualquier variable de `t+h`.
'''
_display(Markdown(texto_d14))

## Registro de la corrida

In [ ]:
duracion = time.time() - t0_proceso
pm.registrar_corrida(
    META / "corridas.csv", "06_poblacion_riesgo",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None},
    len(trimestres_a_procesar), n_filas_total, duracion, COMMIT_GIT,
)
print(f"corrida registrada -- {len(trimestres_a_procesar)} trimestres, {n_filas_total:,} hogares en caché, {duracion:.1f}s")

## Sección 6 — Caída comparable con salida: intensificación (D16/D19/D20)

Implementa D16 (intensificación: sube la proporción de activos informales/
desocupados del hogar, emparejados por sexo+edad entre `t` y `t+h`) y la curación de
eventos de composición de D19 que la distorsionarían sin tratarlos aparte (menores que
entran a trabajar, jubilaciones en hogares mixtos, nacimientos). Misma población que
la salida (hogares con `identidad_confirmada==True`, excluyendo los orígenes D13 en
`h=4`), misma estratificación por decil y el mismo `src/bootstrap.py` que las
secciones 2-3.

**`src/intensificacion.py`** empareja activos (`ESTADO∈{1,2}`) por sexo+edad (D6) y
separa `estado` en `"intensifica"`/`"no_intensifica"`/`"sin_activos_estables"` -- este
último nunca se pisa en silencio. **Tratamiento de independientes (D16, PENDIENTE de
elegir)**: se corren y reportan las 3 variantes (`informal`/`categoria_propia`/
`excluir`), sin recomendar ninguna. **`src/eventos_hogar.py`** separa `se_jubila`
(activo en `t`, `CAT_INAC==1` en `t+h` -- D19, evento propio, no cuenta como
intensificación) y `alta_activo_menor` (por `escolarizado_th`, D19); `nacimiento` viene
de `panel.contar_nacimientos` (D19/D21: usa `CH06==-1`, no `0` -- los menores de 1 año
están codificados así en los datos, no documentado en el codebook).

`CAT_INAC`/`CH10` se traen desde `01_armonizado` igual que `NIVEL_ED` (vía
`pipeline_meta.cargar_individual`) -- no se agregan a `tipado_nucleo.ESQUEMA_NUCLEO`,
mismo criterio que el resto de las variables ad hoc de este notebook.

In [ ]:
import intensificacion as it
import eventos_hogar as eh

COLUMNAS_HOGAR_INTENSIF = COLUMNAS_HOGAR + ["V2"]
COLUMNAS_INDIVIDUAL_INTENSIF = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H"]


def cargar_cat_inac_ch10_nivel_ed(anio, trimestre):
    p = pm.ruta_particion(ARMONIZADO / "individual", "", anio, trimestre)
    if not p.exists():
        return None
    return pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CAT_INAC", "CH10", "NIVEL_ED"])


In [ ]:
filas_pares_caida = []
filas_pares_eventos = []
n_activos_t_con_cat_inac_uno = 0

for h in (1, 4):
    for anio_t, trimestre_t in trimestres_a_procesar:
        if h == 4 and (anio_t, trimestre_t) in ORIGENES_EXCLUIDOS_D13:
            continue
        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
        era = "historica" if es_historico(anio_t, trimestre_t) else "regular"

        p_pares = pm.ruta_particion(PANEL / f"pares_h{h}", "", anio_t, trimestre_t)
        if not p_pares.exists():
            continue
        pares = pd.read_parquet(p_pares, columns=["CODUSU", "AGLOMERADO", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada"])
        pares = pares[pares["identidad_confirmada"] == True]
        if pares.empty:
            continue

        individual_t = pm.cargar_individual(anio_t, trimestre_t, COLUMNAS_INDIVIDUAL_INTENSIF, NUCLEO)
        individual_th = pm.cargar_individual(anio_th, trimestre_th, COLUMNAS_INDIVIDUAL_INTENSIF, NUCLEO)
        hogar_t = pm.cargar_hogar(anio_t, trimestre_t, COLUMNAS_HOGAR_INTENSIF, NUCLEO)
        extra_t = cargar_cat_inac_ch10_nivel_ed(anio_t, trimestre_t)
        extra_th = cargar_cat_inac_ch10_nivel_ed(anio_th, trimestre_th)
        if individual_t is None or individual_th is None or hogar_t is None or extra_t is None or extra_th is None:
            continue
        individual_th_ext = individual_th.merge(extra_th, on=["CODUSU", "NRO_HOGAR", "COMPONENTE"], how="left")

        # Verificación de D19/D20: un activo en t nunca puede tener CAT_INAC==1 en t
        # (CAT_INAC solo toma valor para ESTADO∉{1,2}) -- confirma que
        # `eventos_hogar.calcular_se_jubila` nunca cuenta a quien ya era jubilado.
        estado_t_num = pd.to_numeric(individual_t["ESTADO"], errors="coerce")
        individual_t_verif = individual_t.merge(extra_t[["CODUSU", "NRO_HOGAR", "COMPONENTE", "CAT_INAC"]],
                                                  on=["CODUSU", "NRO_HOGAR", "COMPONENTE"], how="left")
        cat_inac_t_activos = pd.to_numeric(
            individual_t_verif.loc[estado_t_num.isin([1, 2]), "CAT_INAC"], errors="coerce")
        n_activos_t_con_cat_inac_uno += int((cat_inac_t_activos == 1).sum())

        idx_hogar_t = hogar_t.set_index(["CODUSU", "NRO_HOGAR"]).index
        n_miembros_t = individual_t.groupby(["CODUSU", "NRO_HOGAR"]).size()
        decil_t = calcular_decil(hogar_t).to_numpy()
        ancla_con_deficit_t = calcular_componentes_ancla(hogar_t.set_index(["CODUSU", "NRO_HOGAR"]), n_miembros_t)[
            "ancla_con_deficit"].reindex(idx_hogar_t).to_numpy()
        hogar_extra = pd.DataFrame({
            "CODUSU": hogar_t["CODUSU"], "NRO_HOGAR_t": hogar_t["NRO_HOGAR"],
            "decil_t": decil_t, "ancla_con_deficit_t": ancla_con_deficit_t,
        })
        cluster_por_hogar = pares[["CODUSU", "NRO_HOGAR_t", "AGLOMERADO"]].drop_duplicates()
        cluster_por_hogar["cluster_id"] = cluster_por_hogar["CODUSU"].astype(str) + "_" + cluster_por_hogar["AGLOMERADO"].astype(str)

        tipologia_t = eh.calcular_tipologia_perceptores(hogar_t, individual_t).rename(columns={"NRO_HOGAR": "NRO_HOGAR_t"})
        se_jubila = eh.calcular_se_jubila(individual_t_verif, individual_th_ext, pares, h)
        alta_menor = eh.calcular_alta_activo_menor(individual_t, individual_th_ext, pares, h)
        columnas_alta_menor = [c for c in alta_menor.columns if c.startswith("n_alta_activo_menor")]
        alta_menor = alta_menor[["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"] + columnas_alta_menor]
        nacimientos = panel.contar_nacimientos(individual_t, individual_th, pares, h)

        eventos = se_jubila.merge(alta_menor, on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"])
        eventos = eventos.merge(nacimientos, on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"])
        eventos = eventos.merge(hogar_extra, on=["CODUSU", "NRO_HOGAR_t"], how="left")
        eventos = eventos.merge(tipologia_t, on=["CODUSU", "NRO_HOGAR_t"], how="left")
        eventos["h"], eventos["anio_t"], eventos["trimestre_t"], eventos["era"] = h, anio_t, trimestre_t, era
        filas_pares_eventos.append(eventos)

        for tratamiento in it.TRATAMIENTOS_INDEPENDIENTE:
            resultado = it.calcular_intensificacion(individual_t, individual_th, pares, h, tratamiento)
            resultado = resultado.merge(hogar_extra, on=["CODUSU", "NRO_HOGAR_t"], how="left")
            resultado = resultado.merge(cluster_por_hogar[["CODUSU", "NRO_HOGAR_t", "cluster_id"]],
                                         on=["CODUSU", "NRO_HOGAR_t"], how="left")
            resultado["h"], resultado["tratamiento"] = h, tratamiento
            resultado["anio_t"], resultado["trimestre_t"], resultado["era"] = anio_t, trimestre_t, era
            filas_pares_caida.append(resultado)

pares_caida = pd.concat(filas_pares_caida, ignore_index=True)
pares_eventos = pd.concat(filas_pares_eventos, ignore_index=True)
print(f"{len(pares_caida):,} filas hogar-par x tratamiento, {len(pares_eventos):,} filas hogar-par (eventos)")
assert n_activos_t_con_cat_inac_uno == 0
print(f"verificación D19: 0 activos en t con CAT_INAC==1 en los {len(trimestres_a_procesar)} trimestres "
      "-- se_jubila nunca puede contar a quien ya era jubilado en t")


**Tasa de intensificación por trimestre, era y `n_activos_t`** (1/2/3+, D16):
sobre hogares con al menos un activo estable en ambas puntas (se excluye
`sin_activos_estables` del denominador, mismo criterio que `pasa_a_sin_activos` en la
salida -- D14).

In [ ]:
BUCKETS_N_ACTIVOS = {1: "1", 2: "2"}


def bucket_n_activos(n):
    return BUCKETS_N_ACTIVOS.get(n, "3+") if n >= 1 else "0"


pares_caida["bucket_n_activos_t"] = pares_caida["n_activos_t"].map(bucket_n_activos)
determinable = pares_caida["estado"] != "sin_activos_estables"

filas_caida_trimestre = []
for (h, tratamiento, anio_t, trimestre_t, era), sub in pares_caida.groupby(
        ["h", "tratamiento", "anio_t", "trimestre_t", "era"]):
    for bucket in ["1", "2", "3+", "TODOS"]:
        sub_bucket = sub if bucket == "TODOS" else sub[sub["bucket_n_activos_t"] == bucket]
        if sub_bucket.empty:
            continue
        det = sub_bucket[sub_bucket["estado"] != "sin_activos_estables"]
        filas_caida_trimestre.append({
            "h": h, "tratamiento": tratamiento, "anio_t": anio_t, "trimestre_t": trimestre_t, "era": era,
            "bucket_n_activos_t": bucket, "n_hogares": len(sub_bucket),
            "n_sin_activos_estables": int((sub_bucket["estado"] == "sin_activos_estables").sum()),
            "n_intensifica": int((det["estado"] == "intensifica").sum()),
            "tasa_intensificacion": round((det["estado"] == "intensifica").mean() * 100, 2) if len(det) else np.nan,
        })

riesgo_caida_trimestre = pd.DataFrame(filas_caida_trimestre)
riesgo_caida_trimestre.to_csv(META / f"riesgo_caida_trimestre{SUFIJO}.csv", index=False)
print("--- tasa de intensificación por era, tratamiento y n_activos_t (tabla a mostrar) ---")
display(riesgo_caida_trimestre.groupby(["era", "tratamiento", "bucket_n_activos_t"])["tasa_intensificacion"].mean().round(2))


**Intensificación según `ancla_con_deficit_t`**, con IC95% bootstrap de cluster
(`cluster_id = CODUSU+AGLOMERADO`, `src/bootstrap.py` -- D14), pooled y por decil,
estandarizado por decil -- mismas 3 funciones que la salida (secciones 2b-3).

In [ ]:
filas_ancla_bootstrap = []
filas_ancla_por_decil = []
filas_ancla_estandarizada = []

for h in (1, 4):
    for tratamiento in it.TRATAMIENTOS_INDEPENDIENTE:
        sub = pares_caida[(pares_caida["h"] == h) & (pares_caida["tratamiento"] == tratamiento)
                           & (pares_caida["estado"] != "sin_activos_estables")].copy()
        if sub.empty:
            continue
        sub["intensifica"] = sub["estado"] == "intensifica"

        resultado = diferencia_dos_grupos_bootstrap_cluster(
            sub, "cluster_id", "intensifica", "ancla_con_deficit_t", True, False)
        filas_ancla_bootstrap.append({
            "h": h, "tratamiento": tratamiento,
            "tasa_con_deficit": resultado["tasa_a"], "tasa_sin_deficit": resultado["tasa_b"],
            "diferencia": resultado["diferencia"],
            "diferencia_ic95_low": resultado["diferencia_ic95_low"], "diferencia_ic95_high": resultado["diferencia_ic95_high"],
            "n_con_deficit": resultado["n_a"], "n_sin_deficit": resultado["n_b"], "n_clusters": resultado["n_clusters"],
        })

        for decil, sub_decil in sub.dropna(subset=["decil_t"]).groupby("decil_t"):
            resultado_decil = diferencia_dos_grupos_bootstrap_cluster(
                sub_decil, "cluster_id", "intensifica", "ancla_con_deficit_t", True, False)
            filas_ancla_por_decil.append({
                "h": h, "tratamiento": tratamiento, "decil": int(decil),
                "tasa_con_deficit": resultado_decil["tasa_a"], "tasa_sin_deficit": resultado_decil["tasa_b"],
                "diferencia": resultado_decil["diferencia"],
                "diferencia_ic95_low": resultado_decil["diferencia_ic95_low"],
                "diferencia_ic95_high": resultado_decil["diferencia_ic95_high"],
                "n_con_deficit": resultado_decil["n_a"], "n_sin_deficit": resultado_decil["n_b"],
            })

        resultado_std = estandarizar_diferencia_bootstrap(
            sub.dropna(subset=["decil_t"]), "cluster_id", "intensifica", "ancla_con_deficit_t", True, False, "decil_t")
        filas_ancla_estandarizada.append({
            "h": h, "tratamiento": tratamiento,
            "pct_observado_con_deficit": resultado_std["pct_observado_con"],
            "pct_estandarizado_sin_deficit": resultado_std["pct_estandarizado_sin"],
            "diferencia": resultado_std["diferencia"],
            "diferencia_ic95_low": resultado_std["diferencia_ic95_low"],
            "diferencia_ic95_high": resultado_std["diferencia_ic95_high"],
            "n_clusters": resultado_std["n_clusters"], "n_con_deficit": resultado_std["n_con"],
        })

riesgo_caida_ancla_bootstrap = pd.DataFrame(filas_ancla_bootstrap)
riesgo_caida_ancla_bootstrap.to_csv(META / f"riesgo_caida_ancla_bootstrap{SUFIJO}.csv", index=False)
riesgo_caida_ancla_por_decil = pd.DataFrame(filas_ancla_por_decil)
riesgo_caida_ancla_por_decil.to_csv(META / f"riesgo_caida_ancla_por_decil{SUFIJO}.csv", index=False)
riesgo_caida_ancla_estandarizada = pd.DataFrame(filas_ancla_estandarizada)
riesgo_caida_ancla_estandarizada.to_csv(META / f"riesgo_caida_ancla_estandarizada{SUFIJO}.csv", index=False)
print("--- diferencia pooled por ancla_con_deficit (tabla a mostrar) ---")
display(riesgo_caida_ancla_bootstrap)


**Efecto techo (D16/D20)**: un hogar con `share_precario_t==1.0` no puede
"intensificar" bajo esta definición (asimetría ya declarada en D16, no corregida). Se
reporta (a) el % de hogares en el techo por tratamiento y por `ancla_con_deficit_t`, y
(b) la brecha por `ancla_con_deficit_t` (pooled y estandarizada por decil) **solo
entre hogares con `share_precario_t < 1`** -- para ver si el cambio de signo de la
sección anterior sobrevive o es un artefacto del techo.

In [ ]:
filas_techo_pct = []
for h in (1, 4):
    for tratamiento in it.TRATAMIENTOS_INDEPENDIENTE:
        sub = pares_caida[(pares_caida["h"] == h) & (pares_caida["tratamiento"] == tratamiento)
                           & (pares_caida["estado"] != "sin_activos_estables")]
        if sub.empty:
            continue
        fila = {"h": h, "tratamiento": tratamiento,
                "pct_techo_total": round((sub["share_precario_t"] == 1.0).mean() * 100, 2)}
        for ancla_val, nombre in [(True, "con_deficit"), (False, "sin_deficit")]:
            sub_ancla = sub[sub["ancla_con_deficit_t"] == ancla_val]
            fila[f"pct_techo_{nombre}"] = (
                round((sub_ancla["share_precario_t"] == 1.0).mean() * 100, 2) if len(sub_ancla) else np.nan)
        filas_techo_pct.append(fila)

riesgo_caida_techo = pd.DataFrame(filas_techo_pct)
riesgo_caida_techo.to_csv(META / f"riesgo_caida_techo{SUFIJO}.csv", index=False)
print("--- % de hogares en el techo (share_precario_t==1.0), por tratamiento y ancla (tabla a mostrar) ---")
display(riesgo_caida_techo)


In [ ]:
filas_ancla_bootstrap_sin_techo = []
filas_ancla_estandarizada_sin_techo = []

for h in (1, 4):
    for tratamiento in it.TRATAMIENTOS_INDEPENDIENTE:
        sub = pares_caida[(pares_caida["h"] == h) & (pares_caida["tratamiento"] == tratamiento)
                           & (pares_caida["estado"] != "sin_activos_estables")
                           & (pares_caida["share_precario_t"] < 1.0)].copy()
        if sub.empty:
            continue
        sub["intensifica"] = sub["estado"] == "intensifica"

        resultado = diferencia_dos_grupos_bootstrap_cluster(
            sub, "cluster_id", "intensifica", "ancla_con_deficit_t", True, False)
        filas_ancla_bootstrap_sin_techo.append({
            "h": h, "tratamiento": tratamiento,
            "tasa_con_deficit": resultado["tasa_a"], "tasa_sin_deficit": resultado["tasa_b"],
            "diferencia": resultado["diferencia"],
            "diferencia_ic95_low": resultado["diferencia_ic95_low"], "diferencia_ic95_high": resultado["diferencia_ic95_high"],
            "n_con_deficit": resultado["n_a"], "n_sin_deficit": resultado["n_b"], "n_clusters": resultado["n_clusters"],
        })

        resultado_std = estandarizar_diferencia_bootstrap(
            sub.dropna(subset=["decil_t"]), "cluster_id", "intensifica", "ancla_con_deficit_t", True, False, "decil_t")
        filas_ancla_estandarizada_sin_techo.append({
            "h": h, "tratamiento": tratamiento,
            "pct_observado_con_deficit": resultado_std["pct_observado_con"],
            "pct_estandarizado_sin_deficit": resultado_std["pct_estandarizado_sin"],
            "diferencia": resultado_std["diferencia"],
            "diferencia_ic95_low": resultado_std["diferencia_ic95_low"],
            "diferencia_ic95_high": resultado_std["diferencia_ic95_high"],
            "n_clusters": resultado_std["n_clusters"], "n_con_deficit": resultado_std["n_con"],
        })

riesgo_caida_ancla_bootstrap_sin_techo = pd.DataFrame(filas_ancla_bootstrap_sin_techo)
riesgo_caida_ancla_bootstrap_sin_techo.to_csv(META / f"riesgo_caida_ancla_bootstrap_sin_techo{SUFIJO}.csv", index=False)
riesgo_caida_ancla_estandarizada_sin_techo = pd.DataFrame(filas_ancla_estandarizada_sin_techo)
riesgo_caida_ancla_estandarizada_sin_techo.to_csv(META / f"riesgo_caida_ancla_estandarizada_sin_techo{SUFIJO}.csv", index=False)
print("--- brecha pooled por ancla_con_deficit, solo hogares con share_precario_t < 1 (tabla a mostrar) ---")
display(riesgo_caida_ancla_bootstrap_sin_techo)
print("--- brecha estandarizada por decil, solo hogares con share_precario_t < 1 (tabla a mostrar) ---")
display(riesgo_caida_ancla_estandarizada_sin_techo)


**Frecuencia de eventos de composición** (D19), por trimestre, era y decil de
origen -- insumo para decidir en una próxima ronda si cada evento es clase de etiqueta
o marca descriptiva (no se decide en esta entrada).

In [ ]:
COLUMNAS_ALTA_MENOR = [c for c in pares_eventos.columns if c.startswith("n_alta_activo_menor")]
pares_eventos[COLUMNAS_ALTA_MENOR] = pares_eventos[COLUMNAS_ALTA_MENOR].fillna(0)

filas_eventos_trimestre = []
for (h, anio_t, trimestre_t, era), sub in pares_eventos.groupby(["h", "anio_t", "trimestre_t", "era"]):
    for decil, sub_decil in [(None, sub), *list(sub.dropna(subset=["decil_t"]).groupby("decil_t"))]:
        fila = {
            "h": h, "anio_t": anio_t, "trimestre_t": trimestre_t, "era": era,
            "decil_t": "TODOS" if decil is None else int(decil), "n_hogares": len(sub_decil),
            "n_se_jubila": int(sub_decil["n_se_jubila"].sum()),
            "n_nacimientos": int(sub_decil["n_nacimientos"].sum()),
        }
        for col in COLUMNAS_ALTA_MENOR:
            fila[col] = int(sub_decil[col].sum())
        tipologia_counts = sub_decil["tipologia_perceptores"].value_counts()
        for categoria in ("solo_laborales", "solo_previsionales", "mixto", "sin_perceptores"):
            fila[f"n_{categoria}"] = int(tipologia_counts.get(categoria, 0))
        filas_eventos_trimestre.append(fila)

riesgo_eventos_trimestre = pd.DataFrame(filas_eventos_trimestre)
riesgo_eventos_trimestre.to_csv(META / f"riesgo_eventos_trimestre{SUFIJO}.csv", index=False)
print("--- eventos totales por era (TODOS los deciles, tabla a mostrar) ---")
display(riesgo_eventos_trimestre[riesgo_eventos_trimestre["decil_t"] == "TODOS"].groupby("era")[
    ["n_se_jubila", "n_nacimientos", "n_alta_activo_menor"]].sum())
print("--- desglose de alta_activo_menor por situación escolar y tipo, TODOS los deciles (tabla a mostrar) ---")
display(riesgo_eventos_trimestre[riesgo_eventos_trimestre["decil_t"] == "TODOS"].groupby("era")[COLUMNAS_ALTA_MENOR].sum())


**Registro de la corrida de la Sección 6** -- fila aparte en `corridas.csv` (D19/D20),
distinta de la de la Sección 1-5 (celda de arriba): esta sección corre su propio loop
sobre `trimestres_a_procesar` y conviene que su alcance quede trazable por separado.

In [ ]:
duracion_seccion6 = time.time() - t0_proceso
pm.registrar_corrida(
    META / "corridas.csv", "06_poblacion_riesgo_intensificacion",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None},
    len(trimestres_a_procesar), len(pares_caida), duracion_seccion6, COMMIT_GIT,
)
print(f"corrida de intensificación registrada -- {len(pares_caida):,} filas hogar-par x tratamiento, {duracion_seccion6:.1f}s")
